# 🚀 DataMate (Zalo AI Challenge 2026) — Google Colab Runner

Notebook này được thiết kế để chạy trọn vẹn hệ thống **DataMate** trên Google Colab (T4 / L4 / A100 GPU), kết nối trực tiếp với:
- **GitHub Repository công khai**: `https://github.com/Long-uit/datamate.git`
- **Bộ dữ liệu trên Google Drive**: `competition_data/` (`train` & `public_test`)

### Quy trình thực thi (7 bước):
1. **Cấu hình Tập trung**: Chọn nhánh GitHub, đường dẫn Google Drive, mô hình (`7B` hoặc `3B` / `14B`), và tham số đánh giá.
2. **Kiểm tra GPU & Mount Google Drive**: Xác nhận VRAM và kết nối thư mục lưu trữ bền vững.
3. **Clone / Cập nhật GitHub & Cài đặt thư viện**: Tự động kéo code mới nhất và cài đặt `colab/requirements-colab.txt`.
4. **Đồng bộ Dữ liệu về SSD của Colab**: Copy dữ liệu từ Google Drive sang ổ cứng cục bộ `/content/datamate/competition_data` để tăng tốc độ đọc CSV/Excel/SQLite gấp 10 lần.
5. **Kiểm thử nhanh (Unit Tests & Dry-Run)**: Đảm bảo toàn bộ 6 module hoạt động hoàn hảo trước khi nạp GPU.
6. **Đánh giá White-Box & Chấm điểm Penalized Accuracy ($PA$) trên tập `train`**: Chạy `tests/inspect_pipeline.py --batch` kèm biểu đồ trực quan 4-panel.
7. **Suy luận `public_test` (hỗ trợ `--resume`) & Kiểm định `submission.jsonl`**: Lưu trực tiếp từng câu lên Google Drive, chống mất dữ liệu khi ngắt kết nối.

In [ ]:
# ==============================================================================
# CẤU HÌNH TẬP TRUNG (CHỈ CẦN CHỈNH SỬA TẠI Ô NÀY)
# ==============================================================================
REPO_URL = "https://github.com/Long-uit/datamate.git"
REPO_BRANCH = "long-commit"  # Đổi thành "main" nếu bạn đã merge vào nhánh main

# Đường dẫn bộ dữ liệu trên Google Drive (hỗ trợ thư mục hoặc file .zip)
DRIVE_DATA_DIR = "/content/drive/MyDrive/Data_Test/competition_data"
# Thư mục lưu kết quả, logs, biểu đồ và submission.jsonl trên Google Drive
DRIVE_RUN_DIR = "/content/drive/MyDrive/datamate_runs"

# Lựa chọn mô hình LLM:
# - "Qwen/Qwen2.5-Coder-7B-Instruct"  (Khuyên dùng trên Colab T4/L4: chỉ tốn ~5.5GB VRAM ở 4-bit NF4, độ chính xác cao vượt trội)
# - "Qwen/Qwen2.5-Coder-3B-Instruct"  (Nhẹ nhất, tốc độ nhanh, tốn ~2.6GB VRAM)
# - "Qwen/Qwen2.5-Coder-14B-Instruct" (Mạnh nhất <= 15B: dùng trên Colab L4/A100 24GB+ VRAM)
MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"

# Cấu hình đánh giá trên tập Train (Bước 5)
# - Đặt số > 0 (VD: 30): Chạy đánh giá đúng 30 câu trên tập train
# - Đặt 0: Bỏ qua tập train
# - Đặt -1 (hoặc None): Chạy toàn bộ tập train
TRAIN_SAMPLE_SIZE = 30
TRAIN_SEED = 42              # Seed cố định để so sánh công bằng giữa các lần chạy

# Cấu hình suy luận trên tập Public Test (Bước 6)
# - Đặt 0: BỎ QUA không chạy public_test (chỉ chạy tập train)
# - Đặt số > 0 (VD: 200): Chạy từng đợt 200 câu trên public_test kèm --resume
# - Đặt -1 (hoặc None): Chạy TOÀN BỘ tập public_test để lấy file nộp bài hoàn chỉnh
PUBLIC_BATCH_SIZE = 200

# Siêu tham số Pipeline & Cổng Abstention
TAU_STAR = 0.50              # Ngưỡng độ tin cậy tối ưu cho Penalized Accuracy (PA)
TIMEOUT_S = 30               # Thời gian chờ tối đa cho mỗi lần chạy code trong Daemon Sandbox (giây)
MAX_NEW_TOKENS = 1536        # Số token sinh mã tối đa (đảm bảo không bị cắt cụt AST)
PROJECT_ROOT = "/content/datamate"

print(f"✅ Đã nạp cấu hình: MODEL_ID={MODEL_ID} | BRANCH={REPO_BRANCH}")

In [ ]:
# ==============================================================================
# BƯỚC 1: KIỂM TRA GPU & KẾT NỐI GOOGLE DRIVE
# ==============================================================================
import os
import torch
from google.colab import drive

!nvidia-smi
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"✅ GPU sẵn sàng: {gpu_name} ({vram_gb:.1f} GB VRAM)")
else:
    print("⚠️ CẢNH BÁO: Chưa bật GPU! Hãy vào Runtime -> Change runtime type -> T4 GPU.")

drive.mount("/content/drive", force_remount=False)
os.makedirs(DRIVE_RUN_DIR, exist_ok=True)
print(f"✅ Kết quả & Checkpoint sẽ được lưu an toàn tại: {DRIVE_RUN_DIR}")

In [ ]:
# ==============================================================================
# BƯỚC 2: CLONE / CẬP NHẬT MÃ NGUỒN TỪ GITHUB & CÀI ĐẶT THƯ VIỆN
# ==============================================================================
import os
import subprocess

if not os.path.exists(PROJECT_ROOT):
    print(f"🔄 Đang clone repository từ {REPO_URL} (nhánh: {REPO_BRANCH})...")
    ret = subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, PROJECT_ROOT])
    if ret.returncode != 0:
        print(f"⚠️ Không tìm thấy nhánh {REPO_BRANCH}, đang clone nhánh mặc định...")
        !git clone {REPO_URL} {PROJECT_ROOT}
else:
    print(f"🔄 Thư mục {PROJECT_ROOT} đã tồn tại, đang cập nhật code mới nhất từ GitHub...")
    %cd {PROJECT_ROOT}
    !git fetch --all
    !git checkout {REPO_BRANCH} || true
    !git pull origin {REPO_BRANCH} || true

%cd {PROJECT_ROOT}
!git status -sb

print("\n📦 Đang cài đặt các thư viện cần thiết từ colab/requirements-colab.txt...")
!pip install -q -r colab/requirements-colab.txt
!pip install -q -e . --no-deps
print("✅ Hoàn tất cài đặt môi trường DataMate!")

In [ ]:
# ==============================================================================
# BƯỚC 3: ĐỒNG BỘ DỮ LIỆU TỪ GOOGLE DRIVE SANG Ổ CỨNG CỤC BỘ COLAB
# ==============================================================================
import os
import shutil
import zipfile
from pathlib import Path

%cd {PROJECT_ROOT}
local_data_dir = Path(PROJECT_ROOT) / "competition_data"

# Tự động dò tìm đường dẫn dữ liệu trên Google Drive nếu người dùng đặt ở vị trí khác
candidate_drive_paths = [
    Path(DRIVE_DATA_DIR),
    Path("/content/drive/MyDrive/competition_data"),
    Path("/content/drive/MyDrive/Data_Test/competition_data.zip"),
    Path("/content/drive/MyDrive/competition_data.zip"),
]
resolved_source = next((p for p in candidate_drive_paths if p.exists()), None)

if local_data_dir.exists() and (local_data_dir / "public_test" / "questions.jsonl").exists():
    print(f"✅ Dữ liệu đã có sẵn trên SSD cục bộ: {local_data_dir}")
elif resolved_source is None:
    raise FileNotFoundError(
        f"❌ Không tìm thấy dữ liệu tại DRIVE_DATA_DIR={DRIVE_DATA_DIR}. "
        "Hãy kiểm tra lại đường dẫn thư mục competition_data trên Google Drive của bạn!"
    )
else:
    if resolved_source.suffix.lower() == ".zip":
        print(f"📦 Đang giải nén {resolved_source} vào {PROJECT_ROOT}...")
        with zipfile.ZipFile(resolved_source, "r") as zf:
            zf.extractall(PROJECT_ROOT)
    else:
        print(f"📂 Đang copy dữ liệu từ {resolved_source} sang SSD cục bộ {local_data_dir}...")
        shutil.copytree(resolved_source, local_data_dir, dirs_exist_ok=True)
    print("✅ Đồng bộ dữ liệu hoàn tất!")

# Kiểm tra cấu trúc dữ liệu các tập train và public_test
for split in ["train", "public_test"]:
    split_dir = local_data_dir / split
    q_file = split_dir / "questions.jsonl"
    a_file = split_dir / "answers.jsonl"
    d_dir = split_dir / "data"
    if q_file.exists():
        num_q = sum(1 for line in open(q_file, encoding="utf-8") if line.strip())
        num_biz = len([p for p in d_dir.iterdir() if p.is_dir()]) if d_dir.exists() else 0
        has_ans = "Có answers.jsonl" if a_file.exists() else "Không có answers.jsonl (Test split)"
        print(f"  • [{split:<11}] {num_q:>4} câu hỏi | {num_biz:>3} doanh nghiệp | {has_ans}")
    else:
        print(f"  ⚠️ [{split:<11}] Không tìm thấy {q_file}")

In [ ]:
# ==============================================================================
# BƯỚC 4: KIỂM THỬ NHANH HỆ THỐNG (UNIT TESTS & DRY-RUN KHÔNG CẦN GPU)
# ==============================================================================
%cd {PROJECT_ROOT}

print("1️⃣ Chạy kiểm thử đơn vị (Unit Tests)...")
!pytest tests/ -q --ignore=tests/inspect_pipeline.py

print("\n2️⃣ Kiểm thử White-Box 1 câu hỏi ở chế độ Dry-Run...")
!python tests/inspect_pipeline.py \
    --data_dir competition_data/train \
    --random \
    --dry_run

print("\n3️⃣ Kiểm thử luồng nộp bài predict.py (Dry-Run 5 câu)...")
!python predict.py \
    --data_dir competition_data/public_test \
    --output_file /tmp/test_sub_dry.jsonl \
    --log_dir /tmp/test_logs_dry \
    --limit 5 \
    --no_fill_abstain \
    --dry_run
print("✅ Toàn bộ kiểm thử Dry-Run đã vượt qua thành công!")

In [ ]:
# ==============================================================================
# BƯỚC 5: ĐÁNH GIÁ CHẤT LƯỢNG & ĐIỂM PENALIZED ACCURACY (PA) TRÊN TẬP TRAIN
# ==============================================================================
import os
from IPython.display import Image, display

%cd {PROJECT_ROOT}
train_eval_dir = f"{DRIVE_RUN_DIR}/train_eval"
os.makedirs(train_eval_dir, exist_ok=True)

train_pred_file = f"{train_eval_dir}/train_predictions.jsonl"
train_log_dir = f"{train_eval_dir}/logs"
train_report_file = f"{train_eval_dir}/evaluation_report.json"
train_md_file = f"{train_eval_dir}/evaluation_report.md"
train_chart_file = f"{train_eval_dir}/evaluation_dashboard.png"

if TRAIN_SAMPLE_SIZE == 0:
    print("⏭️ TRAIN_SAMPLE_SIZE = 0 -> Bỏ qua đánh giá trên tập TRAIN.")
elif TRAIN_SAMPLE_SIZE is not None and int(TRAIN_SAMPLE_SIZE) > 0:
    print(f"🚀 Bắt đầu đánh giá {TRAIN_SAMPLE_SIZE} câu hỏi trên tập TRAIN với mô hình {MODEL_ID}...")
    !python tests/inspect_pipeline.py \
        --batch \
        --with_llm \
        --data_dir competition_data/train \
        --model_id "{MODEL_ID}" \
        --sample {TRAIN_SAMPLE_SIZE} \
        --seed {TRAIN_SEED} \
        --tau_star {TAU_STAR} \
        --timeout {TIMEOUT_S} \
        --max_new_tokens {MAX_NEW_TOKENS} \
        --no_fill_abstain \
        --resume \
        --output_file "{train_pred_file}" \
        --log_dir "{train_log_dir}" \
        --report_file "{train_report_file}" \
        --md_file "{train_md_file}" \
        --chart_file "{train_chart_file}"
else:
    print(f"🚀 Bắt đầu đánh giá TOÀN BỘ câu hỏi trên tập TRAIN với mô hình {MODEL_ID}...")
    !python tests/inspect_pipeline.py \
        --batch \
        --with_llm \
        --data_dir competition_data/train \
        --model_id "{MODEL_ID}" \
        --tau_star {TAU_STAR} \
        --timeout {TIMEOUT_S} \
        --max_new_tokens {MAX_NEW_TOKENS} \
        --no_fill_abstain \
        --resume \
        --output_file "{train_pred_file}" \
        --log_dir "{train_log_dir}" \
        --report_file "{train_report_file}" \
        --md_file "{train_md_file}" \
        --chart_file "{train_chart_file}"

if TRAIN_SAMPLE_SIZE != 0 and os.path.exists(train_chart_file):
    print("\n📊 Biểu đồ tổng hợp kết quả đánh giá (Evaluation Dashboard):")
    display(Image(filename=train_chart_file))

In [ ]:
# ==============================================================================
# BƯỚC 6: SUY LUẬN CHÍNH THỨC TRÊN PUBLIC TEST (HỖ TRỢ CHECKPOINT --resume)
# ==============================================================================
# Lưu ý: Nhờ cờ --resume và cơ chế flush+fsync theo từng câu, nếu Colab bị ngắt
# kết nối giữa chừng, bạn chỉ cần chạy lại ô này để tiếp tục từ câu tiếp theo!
import os

%cd {PROJECT_ROOT}
pub_out_dir = f"{DRIVE_RUN_DIR}/public_test"
os.makedirs(pub_out_dir, exist_ok=True)

submission_file = f"{pub_out_dir}/submission.jsonl"
pub_log_dir = f"{pub_out_dir}/logs"

if PUBLIC_BATCH_SIZE == 0:
    print("⏭️ PUBLIC_BATCH_SIZE = 0 -> Bỏ qua suy luận trên tập PUBLIC_TEST (chỉ chạy tập Train).")
elif PUBLIC_BATCH_SIZE is not None and int(PUBLIC_BATCH_SIZE) > 0:
    print(f"🚀 Chạy theo đợt {PUBLIC_BATCH_SIZE} câu mới trên public_test (chế độ --resume)...")
    !python predict.py \
        --data_dir competition_data/public_test \
        --model_id "{MODEL_ID}" \
        --output_file "{submission_file}" \
        --log_dir "{pub_log_dir}" \
        --tau_star {TAU_STAR} \
        --timeout {TIMEOUT_S} \
        --limit {PUBLIC_BATCH_SIZE} \
        --no_fill_abstain \
        --resume
else:
    print("🚀 Chạy toàn bộ các câu còn lại trên public_test và điền đầy đủ file nộp bài...")
    !python predict.py \
        --data_dir competition_data/public_test \
        --model_id "{MODEL_ID}" \
        --output_file "{submission_file}" \
        --log_dir "{pub_log_dir}" \
        --tau_star {TAU_STAR} \
        --timeout {TIMEOUT_S} \
        --resume

In [ ]:
# ==============================================================================
# BƯỚC 7: HOÀN TẤT, KIỂM ĐỊNH TOÀN VẸN & TẢI FILE NỘP BÀI SUBMISSION.JSONL
# ==============================================================================
import json
from collections import Counter
from pathlib import Path
from datamate.output_abstention.submission import validate_submission

%cd {PROJECT_ROOT}
submission_path = Path(f"{DRIVE_RUN_DIR}/public_test/submission.jsonl")
questions_path = Path("competition_data/public_test/questions.jsonl")

# Kiểm tra xem đã chạy đủ số câu chưa; nếu thiếu có thể chạy predict.py --dry_run --resume để điền abstain phần còn lại
all_qids = [json.loads(line)["id"] for line in open(questions_path, encoding="utf-8") if line.strip()]
done_records = {}
if submission_path.exists():
    for line in open(submission_path, encoding="utf-8"):
        if line.strip():
            obj = json.loads(line)
            done_records[obj["id"]] = obj

print(f"📌 Tiến độ hiện tại: {len(done_records)} / {len(all_qids)} câu hỏi ({100*len(done_records)/len(all_qids):.1f}%)")

if len(done_records) < len(all_qids):
    print("⚠️ Chưa chạy hết toàn bộ câu hỏi. Nếu bạn muốn xuất file nộp bài ngay (điền 'abstain' cho các câu chưa chạy), bỏ comment dòng dưới:")
    print(f'# !python predict.py --data_dir competition_data/public_test --output_file "{submission_path}" --resume --dry_run')
else:
    errors = validate_submission(submission_path, questions_path)
    if not errors:
        print("🎉 THẨM ĐỊNH THÀNH CÔNG: File submission.jsonl hợp lệ 100% theo quy chuẩn Ban Tổ Chức!")
    else:
        print(f"❌ Phát hiện {len(errors)} lỗi định dạng:")
        for err in errors[:10]:
            print(f"  - {err}")

if done_records:
    type_counts = Counter(r.get("type", "unknown") for r in done_records.values())
    print("\n📊 Phân bố các kiểu đáp án trong submission.jsonl:")
    for t_name, cnt in type_counts.most_common():
        print(f"  • {t_name:<15}: {cnt:>5} ({100*cnt/len(done_records):5.1f}%)")

# Bỏ comment 2 dòng dưới nếu muốn tải trực tiếp file submission.jsonl về máy tính:
# from google.colab import files
# files.download(str(submission_path))